In [0]:
import logging
import uuid
import random
from pyspark.sql import Row

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

RAW_SOCIAL_PATH = "abfss://raw@cryptodl.dfs.core.windows.net/JSON_Streaming_Data_Source_2/"
S1_PATH         = "abfss://bronzelayer@cryptodl.dfs.core.windows.net/Batch_Data_Source_1/"

POSTS = [
    "{} adoption continues to increase among institutions",
    "{} trading volume reaches monthly high",
    "{} ecosystem sees strong growth this week",
    "{} investors remain bullish despite volatility",
    "{} faces selling pressure after market correction",
    "{} trading activity declines this week",
    "{} investors express concern over volatility",
    "{} experiences short-term bearish momentum",
    "{} trades within expected range",
    "{} market remains stable",
    "{} investors wait for new signals",
    "{} volume remains unchanged"
]

log.info("Starting Source 2 — Social Media Generation")
log.info(f"Target path: {RAW_SOCIAL_PATH}")

log.info("Reading S1 combos from Bronze...")
s1 = spark.read.parquet(S1_PATH)

# orderBy + limit = same 50 combos every single run — no randomness
sampled = (
    s1.select("Symbol", "trade_date")
      .distinct()
      .orderBy("Symbol", "trade_date")
      .limit(50)
      .collect()
)
log.info(f"Sampled: {len(sampled)} combos — deterministic, same every run")

events = []
for row in sampled:
    symbol   = row["Symbol"]
    date_str = str(row["trade_date"])[:10]
    events.append(
        Row(
            post_id          = str(uuid.uuid4()),
            symbol           = symbol,
            trade_date       = date_str,
            source_type      = random.choice(["twitter","reddit","telegram","news"]),
            engagement_score = random.randint(100, 5000),
            text             = random.choice(POSTS).format(symbol),
            event_timestamp  = f"{date_str}T{random.randint(0,23):02d}:{random.randint(0,59):02d}:00"
        )
    )

log.info(f"Generated {len(events)} social events")
log.info(f"Sample — symbol: {events[0].symbol} | date: {events[0].trade_date}")

log.info("Writing to Bronze (append mode)...")
(
    spark.createDataFrame(events)
         .coalesce(1)
         .write
         .mode("append")
         .json(RAW_SOCIAL_PATH)
)

log.info(f"Done! {len(events)} social events written to Bronze")
log.info("=" * 50) 

In [0]:
RAW_SOCIAL_PATH = "abfss://raw@cryptodl.dfs.core.windows.net/JSON_Streaming_Data_Source_2/"
df = spark.read.json(RAW_SOCIAL_PATH)

df.groupBy(
    "symbol",
    "event_timestamp"
).count().filter("count > 1").show()